# The Ultimate Titanic Survival Guide: From Zero to Top 5%

**A comprehensive walkthrough of the Titanic competition with insights that actually matter.**

This notebook takes you from raw data to a top-performing submission, explaining not just *what* to do but *why* each step matters. Perfect for beginners and useful for experienced Kagglers looking for new techniques.

## What Makes This Notebook Different

| Section | What You'll Learn |
|---------|-------------------|
| EDA | Visualization techniques that reveal hidden patterns |
| Feature Engineering | 15+ features that actually improve your score |
| Modeling | 5 different algorithms compared fairly |
| Ensembling | How to combine models like competition winners |
| Submission | Common pitfalls and how to avoid them |

**Let's begin!**

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Advanced models
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available, will skip XGB models")

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12']

print("Libraries loaded successfully!")

## 1. Data Loading & First Look

Before diving into complex analysis, let's understand what we're working with.

In [ ]:
# Load data
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

# Store PassengerId for submission
test_ids = test['PassengerId'].copy()

print(f"Training set: {train.shape[0]} passengers, {train.shape[1]} features")
print(f"Test set: {test.shape[0]} passengers, {test.shape[1]} features")
print(f"\nTarget distribution:")
print(train['Survived'].value_counts(normalize=True).round(3))

In [ ]:
# Quick overview
train.head()

In [ ]:
# Data types and missing values
def data_overview(df, name='Dataset'):
    """Comprehensive data overview."""
    print(f"\n{'='*50}")
    print(f"{name} Overview")
    print(f"{'='*50}")
    
    overview = pd.DataFrame({
        'Type': df.dtypes,
        'Non-Null': df.count(),
        'Null': df.isnull().sum(),
        'Null %': (df.isnull().sum() / len(df) * 100).round(1),
        'Unique': df.nunique()
    })
    
    return overview

data_overview(train, 'Training Data')

### Key Observations

**Missing Values:**
- `Age`: ~20% missing - significant, need smart imputation
- `Cabin`: ~77% missing - too much to impute normally, but the missingness itself is informative!
- `Embarked`: Only 2 missing - easy to handle

**Feature Types:**
- Numerical: Age, SibSp, Parch, Fare
- Categorical: Sex, Embarked, Pclass
- Text: Name, Ticket, Cabin (need feature extraction)

## 2. Exploratory Data Analysis (EDA)

Let's visualize the data to find patterns that will guide our feature engineering.

In [ ]:
# Survival rate by key features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Sex
sns.barplot(data=train, x='Sex', y='Survived', ax=axes[0,0], palette=colors)
axes[0,0].set_title('Survival by Sex\n(Women had 3x higher survival rate)', fontsize=11)
axes[0,0].set_ylabel('Survival Rate')

# 2. Pclass
sns.barplot(data=train, x='Pclass', y='Survived', ax=axes[0,1], palette=colors)
axes[0,1].set_title('Survival by Class\n(1st class > 2nd class > 3rd class)', fontsize=11)
axes[0,1].set_ylabel('Survival Rate')

# 3. Age distribution
axes[0,2].hist([train[train['Survived']==1]['Age'].dropna(), 
                train[train['Survived']==0]['Age'].dropna()],
               bins=30, label=['Survived', 'Died'], alpha=0.7, color=[colors[2], colors[1]])
axes[0,2].set_title('Age Distribution by Survival\n(Children had higher survival)', fontsize=11)
axes[0,2].set_xlabel('Age')
axes[0,2].legend()

# 4. Embarked
sns.barplot(data=train, x='Embarked', y='Survived', ax=axes[1,0], palette=colors)
axes[1,0].set_title('Survival by Embarkation Port\n(C=Cherbourg had highest rate)', fontsize=11)
axes[1,0].set_ylabel('Survival Rate')

# 5. Family size
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
sns.barplot(data=train, x='FamilySize', y='Survived', ax=axes[1,1], palette='viridis')
axes[1,1].set_title('Survival by Family Size\n(Small families (2-4) did best)', fontsize=11)
axes[1,1].set_ylabel('Survival Rate')

# 6. Fare distribution
axes[1,2].hist([train[train['Survived']==1]['Fare'].dropna(), 
                train[train['Survived']==0]['Fare'].dropna()],
               bins=50, label=['Survived', 'Died'], alpha=0.7, color=[colors[2], colors[1]])
axes[1,2].set_title('Fare Distribution by Survival\n(Higher fare = higher survival)', fontsize=11)
axes[1,2].set_xlabel('Fare')
axes[1,2].set_xlim(0, 200)
axes[1,2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# The most important insight: Sex + Class interaction
pivot = train.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='RdYlGn', center=0.5, ax=ax)
ax.set_title('Survival Rate: Sex x Class\n(This single insight can get you to top 20%)', fontsize=12)
plt.show()

print("\nKey Insight: A 1st class woman had 97% survival rate vs 14% for 3rd class men")
print("This Sex-Class interaction is the single most predictive feature!")

### EDA Insights Summary

| Feature | Insight | Action |
|---------|---------|--------|
| Sex | Women survived at 3x the rate of men | Keep as-is, very predictive |
| Pclass | Clear hierarchy in survival | Keep as-is, create interactions |
| Age | Children survived more, but many missing | Impute carefully, create age groups |
| Fare | Higher fare = higher survival | Log transform, handle outliers |
| Family | Size 2-4 optimal | Create FamilySize and IsAlone features |
| Embarked | C port had higher survival | Likely correlated with class/fare |
| Cabin | 77% missing, but missingness is informative! | Create HasCabin feature |

## 3. Feature Engineering

This is where competitions are won or lost. Let's create features that capture the patterns we discovered.

In [ ]:
def engineer_features(df):
    """Apply all feature engineering steps."""
    data = df.copy()
    
    # =========================================================================
    # FEATURE 1: Title extraction from Name
    # Names contain titles like 'Mr.', 'Mrs.', 'Master.' which indicate
    # social status, gender, and sometimes age
    # =========================================================================
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    
    # Group rare titles
    title_mapping = {
        'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
        'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
        'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
        'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
        'Capt': 'Rare', 'Sir': 'Rare'
    }
    data['Title'] = data['Title'].map(title_mapping).fillna('Rare')
    
    # =========================================================================
    # FEATURE 2: Family features
    # Family dynamics strongly affect survival
    # =========================================================================
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    
    # Family size categories
    data['FamilySizeGroup'] = pd.cut(data['FamilySize'], 
                                      bins=[0, 1, 4, 11],
                                      labels=['Alone', 'Small', 'Large'])
    
    # =========================================================================
    # FEATURE 3: Cabin features
    # Missing cabin info is itself informative (likely lower class)
    # =========================================================================
    data['HasCabin'] = data['Cabin'].notna().astype(int)
    
    # Extract cabin deck (first letter)
    data['Deck'] = data['Cabin'].str[0].fillna('Unknown')
    
    # =========================================================================
    # FEATURE 4: Fare features
    # Handle outliers and missing values
    # =========================================================================
    # Fill missing fare with median by Pclass
    data['Fare'] = data.groupby('Pclass')['Fare'].transform(
        lambda x: x.fillna(x.median())
    )
    
    # Log transform to handle skewness
    data['FareLog'] = np.log1p(data['Fare'])
    
    # Fare per person (for families sharing tickets)
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    
    # =========================================================================
    # FEATURE 5: Age features
    # Smart imputation using Title (Master = child, Mr = adult, etc.)
    # =========================================================================
    # Impute age by Title median
    data['Age'] = data.groupby('Title')['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    # Fallback to overall median
    data['Age'] = data['Age'].fillna(data['Age'].median())
    
    # Age groups
    data['AgeGroup'] = pd.cut(data['Age'],
                              bins=[0, 12, 18, 35, 60, 100],
                              labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])
    
    # Is child (special treatment on Titanic)
    data['IsChild'] = (data['Age'] < 12).astype(int)
    
    # =========================================================================
    # FEATURE 6: Embarked
    # =========================================================================
    data['Embarked'] = data['Embarked'].fillna('S')  # Most common
    
    # =========================================================================
    # FEATURE 7: Ticket features
    # Passengers with same ticket number traveled together
    # =========================================================================
    ticket_counts = data['Ticket'].value_counts()
    data['TicketCount'] = data['Ticket'].map(ticket_counts)
    
    # =========================================================================
    # FEATURE 8: Interaction features
    # Capture non-linear relationships
    # =========================================================================
    data['Sex_Pclass'] = data['Sex'] + '_' + data['Pclass'].astype(str)
    
    # Woman or child in 1st/2nd class (very high survival)
    data['WomanOrChildUpperClass'] = (
        ((data['Sex'] == 'female') | (data['Age'] < 12)) & 
        (data['Pclass'].isin([1, 2]))
    ).astype(int)
    
    # Man in 3rd class (very low survival)
    data['ManThirdClass'] = (
        (data['Sex'] == 'male') & (data['Pclass'] == 3)
    ).astype(int)
    
    return data

# Apply feature engineering to both datasets
train_fe = engineer_features(train)
test_fe = engineer_features(test)

print(f"Features after engineering: {train_fe.shape[1]}")
print(f"\nNew features created:")
new_features = ['Title', 'FamilySize', 'IsAlone', 'FamilySizeGroup', 'HasCabin', 'Deck',
                'FareLog', 'FarePerPerson', 'AgeGroup', 'IsChild', 'TicketCount',
                'Sex_Pclass', 'WomanOrChildUpperClass', 'ManThirdClass']
for f in new_features:
    print(f"  - {f}")

In [ ]:
# Verify our best features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Title survival
title_surv = train_fe.groupby('Title')['Survived'].mean().sort_values()
title_surv.plot(kind='barh', ax=axes[0], color=colors[0])
axes[0].set_title('Survival by Title')
axes[0].set_xlabel('Survival Rate')

# Has Cabin
cabin_surv = train_fe.groupby('HasCabin')['Survived'].mean()
cabin_surv.plot(kind='bar', ax=axes[1], color=[colors[1], colors[2]])
axes[1].set_title('Survival by Has Cabin\n(Missing cabin info = lower survival)')
axes[1].set_xticklabels(['No Cabin', 'Has Cabin'], rotation=0)
axes[1].set_ylabel('Survival Rate')

# Woman/Child Upper Class
special_surv = train_fe.groupby('WomanOrChildUpperClass')['Survived'].mean()
special_surv.plot(kind='bar', ax=axes[2], color=[colors[1], colors[2]])
axes[2].set_title('Woman/Child in 1st/2nd Class\n(Our most predictive engineered feature)')
axes[2].set_xticklabels(['Others', 'Woman/Child Upper'], rotation=0)
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

Now let's prepare the data for modeling.

In [ ]:
def prepare_for_modeling(train_df, test_df):
    """Prepare data for machine learning models."""
    
    # Features to use
    numeric_features = ['Age', 'Fare', 'FareLog', 'FarePerPerson', 'FamilySize', 'TicketCount']
    binary_features = ['IsAlone', 'HasCabin', 'IsChild', 'WomanOrChildUpperClass', 'ManThirdClass']
    categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'Deck']
    
    # Combine train and test for consistent encoding
    combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    
    # One-hot encode categorical features
    combined_encoded = pd.get_dummies(combined, columns=categorical_features, drop_first=True)
    
    # Split back
    train_encoded = combined_encoded.iloc[:len(train_df)].copy()
    test_encoded = combined_encoded.iloc[len(train_df):].copy()
    
    # Get feature columns (exclude identifiers and target)
    exclude_cols = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin', 
                    'FamilySizeGroup', 'AgeGroup', 'Sex_Pclass']
    feature_cols = [c for c in train_encoded.columns if c not in exclude_cols]
    
    # Prepare X and y
    X_train = train_encoded[feature_cols].copy()
    y_train = train_encoded['Survived'].copy()
    X_test = test_encoded[feature_cols].copy()
    
    # Handle any remaining NaN (shouldn't be many)
    X_train = X_train.fillna(X_train.median())
    X_test = X_test.fillna(X_train.median())
    
    # Scale numeric features
    scaler = StandardScaler()
    numeric_cols = [c for c in numeric_features if c in X_train.columns]
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
    
    return X_train, y_train, X_test, feature_cols

X_train, y_train, X_test, feature_cols = prepare_for_modeling(train_fe, test_fe)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeatures ({len(feature_cols)}): {feature_cols[:10]}...")

## 5. Model Training & Comparison

Let's train multiple models and compare them fairly using cross-validation.

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, 
                                            min_samples_split=4, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                                    learning_rate=0.1, random_state=42),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42),
}

if HAS_XGB:
    models['XGBoost'] = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                      subsample=0.8, colsample_bytree=0.8,
                                      random_state=42, use_label_encoder=False,
                                      eval_metric='logloss')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
print("Cross-Validation Results (5-fold)")
print("="*50)

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    results[name] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"{name:22s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(results.keys())
means = [results[m]['mean'] for m in model_names]
stds = [results[m]['std'] for m in model_names]

bars = ax.barh(model_names, means, xerr=stds, color=colors[:len(model_names)], alpha=0.8)
ax.set_xlabel('Accuracy')
ax.set_title('Model Comparison (5-fold CV)')
ax.set_xlim(0.75, 0.90)

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(mean + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{mean:.3f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

## 6. Ensemble Model

Top Kagglers almost always use ensembles. Let's combine our best models.

In [ ]:
# Create ensemble of top models
ensemble_models = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=8, 
                                   min_samples_split=4, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                       learning_rate=0.1, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
]

if HAS_XGB:
    ensemble_models.append(
        ('xgb', XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, random_state=42,
                             use_label_encoder=False, eval_metric='logloss'))
    )

# Voting ensemble
ensemble = VotingClassifier(estimators=ensemble_models, voting='soft')

# Evaluate ensemble
ensemble_scores = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring='accuracy')
print(f"Ensemble CV Score: {ensemble_scores.mean():.4f} (+/- {ensemble_scores.std():.4f})")
print(f"\nThis is our best model - let's use it for the final submission!")

### Segment-Level Validation Audit

Average cross-validation score is useful, but leaderboard gains usually come from finding the segments where the model still fails. This audit highlights which passenger groups deserve the next round of feature engineering.

In [ ]:
ensemble_oof = cross_val_predict(ensemble, X_train, y_train, cv=cv, method='predict')

segment_audit = train_fe[['Sex', 'Pclass', 'Embarked', 'IsAlone', 'AgeGroup']].copy()
segment_audit['actual'] = y_train.values
segment_audit['predicted'] = ensemble_oof
segment_audit['correct'] = (segment_audit['actual'] == segment_audit['predicted']).astype(int)

segment_summary = (
    segment_audit
    .groupby(['Sex', 'Pclass'])
    .agg(passengers=('correct', 'size'), accuracy=('correct', 'mean'), survival_rate=('actual', 'mean'))
    .sort_values(['accuracy', 'passengers'], ascending=[True, False])
)

display(segment_summary.head(10).round(3))
print("\nWeakest segment to investigate first:")
display(segment_summary.head(1).round(3))


In [ ]:
# Feature importance from Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis', ax=ax)
ax.set_title('Top 15 Most Important Features\n(Random Forest Feature Importance)')
plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("- Title and Sex-related features dominate (as expected from EDA)")
print("- Our engineered features rank highly!")
print("- Fare and Age are important but not dominant")

## 7. Final Submission

Train on full data and create submission.

In [ ]:
# Train ensemble on full training data
ensemble.fit(X_train, y_train)

# Make predictions
predictions = ensemble.predict(X_test)

# Create submission
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': predictions.astype(int)
})

# Verify submission
print("Submission Preview:")
print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"Predicted survival rate: {predictions.mean():.2%}")
print(f"Training survival rate: {y_train.mean():.2%}")

# Save submission
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to 'submission.csv'")

In [ ]:
# Submission sanity checks
print("Submission QA Checklist")
print("-" * 40)
print(f"Rows match Kaggle test set : {len(submission) == len(test_ids)}")
print(f"PassengerId unique         : {submission['PassengerId'].is_unique}")
print(f"No missing predictions     : {submission['Survived'].isnull().sum() == 0}")
print(f"Binary labels only         : {set(submission['Survived'].unique()).issubset({0, 1})}")
print(f"CSV file ready             : {Path('submission.csv').exists()}")


In [ ]:
# Final visualization of predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training distribution
axes[0].pie(y_train.value_counts().values, labels=['Died', 'Survived'],
            autopct='%1.1f%%', colors=[colors[1], colors[2]])
axes[0].set_title('Training Data Distribution')

# Prediction distribution
pred_counts = pd.Series(predictions).value_counts().sort_index()
axes[1].pie(pred_counts.values, labels=['Died', 'Survived'],
            autopct='%1.1f%%', colors=[colors[1], colors[2]])
axes[1].set_title('Test Predictions Distribution')

plt.tight_layout()
plt.show()

print("\nPredictions look reasonable - similar distribution to training data!")

## Summary & Key Takeaways

### What We Learned

1. **EDA First**: Understanding the data revealed that Sex + Class interaction was the most important factor

2. **Feature Engineering Matters**: Our engineered features like `WomanOrChildUpperClass` and `Title` significantly improved model performance

3. **Missing Values Are Features**: The missing `Cabin` information itself was predictive (no cabin = lower class = lower survival)

4. **Ensemble For The Win**: Combining multiple models gave us more robust predictions than any single model

### Score Progression

| Approach | Expected Score |
|----------|----------------|
| Gender-only baseline | ~0.76 |
| Basic features | ~0.77 |
| + Feature Engineering | ~0.80 |
| + Ensemble | ~0.82 |

### Next Steps to Improve

- **Hyperparameter Tuning**: Use GridSearchCV or Optuna for systematic tuning
- **More Feature Engineering**: Try polynomial features, feature selection
- **Stacking**: More sophisticated ensemble method
- **Neural Networks**: Try a simple MLP

---

**If this notebook helped you, please upvote!** 

Connect with me: [GitHub](https://github.com/gr8monk3ys) | [HuggingFace](https://huggingface.co/gr8monk3ys)

Happy Kaggling!

## Portfolio Quality Addendum

### Objective
Build a strong, interpretable baseline for survival prediction on Titanic data.

### Data
Passenger demographics, fare, cabin, and family context with missing-value patterns.

### Method
Combine domain-driven feature engineering with robust baseline classifiers and validation.

### Evaluation
Use cross-validated accuracy/F1 and confusion analysis for model selection.

### Insight and Trade-off
- Insight: Family and title-derived features consistently outperform raw categorical fields.
- Because social structure encoded in titles and family size captures latent risk factors.
- Therefore prioritize domain-informed feature synthesis before advanced models.
- Trade-off: richer engineered features improve score but reduce immediate interpretability.
- Limitation: small dataset size can inflate fold variance.

## Conclusion and Next Steps

### Summary
Most improvements come from targeted feature design and careful validation, not model novelty.

### Next Steps
1. Run permutation importance to verify feature robustness.
2. Benchmark calibrated probability outputs for thresholded decisions.
3. Add error-case narratives for misclassified passenger groups.